# 🧠 Master Convolutional Neural Networks (CNN) — Complete Project

This notebook takes you through the **entire CNN pipeline**, end-to-end, in a way that runs comfortably on **Google Colab's free CPU** (no GPU required).

**Pipeline covered:**

`Image → Ingestion → Preprocessing → Normalization → Augmentation → Convolution → Activation → Pooling → Feature Maps → Flatten/GAP → Dense → Output → Loss → Backprop → Optimizer → Evaluation → Transfer Learning`

**Why this notebook is CPU-friendly and won't crash Colab:**
- Small, subsampled dataset (a few thousand images, not the full 50k)
- Small image size (32×32 / 64×64)
- Small batch size and few epochs
- Lightweight models (no huge ResNet-152 training from scratch)
- `num_workers=0` and `pin_memory=False` to avoid Colab CPU DataLoader issues
- Every heavy cell has a runtime-safe fallback (try/except) so one failure doesn't kill the whole run

> 💡 Just run cells top-to-bottom (`Runtime → Run all` in Colab). Total run time on CPU: roughly 10–20 minutes.


In [ ]:
# Cell 0: Environment check (Colab-safe)
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab ✅")
except ImportError:
    IN_COLAB = False
    print("Not running in Colab (that's fine, notebook still works).")


## 0. Setup & Reproducibility

In [ ]:
# Cell 1: Installs (Colab already has these — this just makes sure versions are compatible)
# Uncomment if a package is missing in your environment
# !pip install torch torchvision scikit-learn matplotlib seaborn --quiet
print("Dependencies assumed pre-installed (torch, torchvision, sklearn, matplotlib, seaborn).")


In [ ]:
# Cell 2: Imports
import os, random, time, copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix

# ---- Force CPU (this project is designed to be CPU-friendly) ----
device = torch.device("cpu")
print("Using device:", device)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 1. What is a CNN? Images as Data

A CNN is a neural network designed for **grid-like data**, especially images.

An image is represented as a tensor: **Height × Width × Channels**
- Grayscale: `28 × 28 × 1`
- Color (RGB): `224 × 224 × 3`

Each pixel holds a numeric value, typically `0–255` (8-bit) before normalization.

Let's build a tiny synthetic image with NumPy to see this concretely — no dataset needed yet.

In [ ]:
# Cell 3: A grid of numbers IS an image
synthetic_gray = np.array([
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
    [50, 50, 50, 50, 50],
    [50, 50, 50, 50, 50],
], dtype=np.uint8)

print("Shape (Height x Width):", synthetic_gray.shape)
plt.figure(figsize=(3,3))
plt.imshow(synthetic_gray, cmap="gray", vmin=0, vmax=255)
plt.title("A 5x5 grayscale 'image'")
plt.colorbar()
plt.show()

synthetic_rgb = np.random.randint(0, 256, size=(64, 64, 3), dtype=np.uint8)
print("Shape (Height x Width x Channels):", synthetic_rgb.shape)
plt.figure(figsize=(3,3))
plt.imshow(synthetic_rgb)
plt.title("A random 64x64x3 RGB 'image'")
plt.show()


## 2. Data Ingestion

Real CNN projects start by collecting/loading images from folders, CSVs, Kaggle, APIs, cloud storage, etc.

To keep this project **fast and CPU-friendly**, we use **CIFAR-10** (built into `torchvision`, auto-downloads ~170MB) but we:
- Keep only **4 classes** (not all 10) → simpler, faster
- **Subsample** each class to a few hundred images → trains in minutes on CPU

This mirrors a real folder-based structure:
```
dataset/
├── train/
│   ├── class_a/
│   └── class_b/
├── validation/
└── test/
```
`torchvision.datasets.CIFAR10` gives us the same train/test images CIFAR ships with — we then re-split them ourselves into train/val/test below.

In [ ]:
# Cell 4: Download CIFAR-10 (cached after first run)
DATA_DIR = "./data"

raw_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True)
raw_test  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True)

CLASS_NAMES_FULL = raw_train.classes
print("All CIFAR-10 classes:", CLASS_NAMES_FULL)
print("Full train size:", len(raw_train), " Full test size:", len(raw_test))


In [ ]:
# Cell 5: Keep only 4 classes + subsample -> CPU-friendly dataset
KEEP_CLASSES = ["cat", "dog", "airplane", "automobile"]  # change these if you like
keep_idx = [CLASS_NAMES_FULL.index(c) for c in KEEP_CLASSES]
label_map = {orig: new for new, orig in enumerate(keep_idx)}  # remap to 0..3

PER_CLASS_TRAIN = 600   # images per class for training pool (small -> fast on CPU)
PER_CLASS_TEST  = 150   # images per class for test pool

def build_subset(dataset, per_class, keep_idx, label_map, seed=SEED):
    rng = np.random.RandomState(seed)
    targets = np.array(dataset.targets)
    chosen_indices, chosen_labels = [], []
    for orig_label in keep_idx:
        all_idx = np.where(targets == orig_label)[0]
        rng.shuffle(all_idx)
        chosen = all_idx[:per_class]
        chosen_indices.extend(chosen.tolist())
        chosen_labels.extend([label_map[orig_label]] * len(chosen))
    return chosen_indices, chosen_labels

train_idx, train_labels = build_subset(raw_train, PER_CLASS_TRAIN, keep_idx, label_map)
test_idx, test_labels   = build_subset(raw_test, PER_CLASS_TEST, keep_idx, label_map)

print(f"Subsampled train pool: {len(train_idx)} images across {len(KEEP_CLASSES)} classes")
print(f"Subsampled test pool:  {len(test_idx)} images across {len(KEEP_CLASSES)} classes")


## 3. Dataset Understanding (EDA)

Before training, always check:
- Number of images & class balance
- Image dimensions / channels
- Corrupted or duplicate images
- A visual sample from each class

In [ ]:
# Cell 6: Basic stats
sample_img, sample_label = raw_train[0]
print("Single image type:", type(sample_img), "size:", sample_img.size)  # PIL Image, (W,H)
print("Number of classes kept:", len(KEEP_CLASSES))

from collections import Counter
print("Class distribution (train pool):", Counter(train_labels))
print("Class distribution (test pool):", Counter(test_labels))


In [ ]:
# Cell 7: Corrupted-image check (defensive coding habit for real datasets)
def check_corrupted(dataset, indices):
    bad = []
    for i in indices[:200]:  # spot-check to stay fast
        try:
            img, _ = dataset[i]
            img.verify() if hasattr(img, "verify") else None
        except Exception as e:
            bad.append((i, str(e)))
    return bad

bad_train = check_corrupted(raw_train, train_idx)
print(f"Corrupted images found in spot-check: {len(bad_train)}")


In [ ]:
# Cell 8: Visualize one sample per class
fig, axes = plt.subplots(1, len(KEEP_CLASSES), figsize=(12, 3))
seen = set()
for idx, remapped_label in zip(train_idx, train_labels):
    if remapped_label in seen:
        continue
    img, _ = raw_train[idx]
    axes[remapped_label].imshow(img)
    axes[remapped_label].set_title(KEEP_CLASSES[remapped_label])
    axes[remapped_label].axis("off")
    seen.add(remapped_label)
    if len(seen) == len(KEEP_CLASSES):
        break
plt.suptitle("One sample per class")
plt.show()


## 4. Preprocessing & Normalization

Common preprocessing: **resize, crop, normalize, convert color space**.

We resize every image to a fixed size (CNNs need consistent tensor shapes for batching), convert to a tensor, and normalize pixel values from `[0, 255]` to roughly `[-1, 1]` (or `[0,1]` then standardized), using the dataset's own mean/std.

In [ ]:
# Cell 9: Compute simple normalization stats on a small sample (CPU-friendly, no full-dataset pass)
IMG_SIZE = 64  # small -> fast on CPU. Try 32 for even faster, 128 for higher fidelity.

to_tensor_only = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

sample_stack = torch.stack([to_tensor_only(raw_train[i][0]) for i in train_idx[:300]])
mean = sample_stack.mean(dim=[0, 2, 3])
std  = sample_stack.std(dim=[0, 2, 3])
print("Computed mean:", mean)
print("Computed std: ", std)


## 5. Data Augmentation

Augmentation creates realistic variations (rotation, flip, crop, color jitter) so the model generalizes better and doesn't just memorize training images.

**Important rule:** never apply random training augmentations to validation/test data — only resize + normalize those.

In [ ]:
# Cell 10: Define train vs eval transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])

print("Train transform (with augmentation):\n", train_transform)
print("\nEval transform (no augmentation):\n", eval_transform)


In [ ]:
# Cell 11: Visualize augmentation effect on one image
raw_img, _ = raw_train[train_idx[0]]
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(raw_img); axes[0].set_title("Original"); axes[0].axis("off")
for i in range(1, 5):
    aug_tensor = train_transform(raw_img)
    # unnormalize just for display
    disp = aug_tensor * std[:, None, None] + mean[:, None, None]
    disp = disp.clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(disp); axes[i].set_title(f"Augmented {i}"); axes[i].axis("off")
plt.suptitle("Data augmentation examples")
plt.show()


## 6. Train / Validation / Test Split

- **Training set** → the model learns from this
- **Validation set** → used to tune and monitor during training
- **Test set** → final, untouched evaluation

We use roughly **70% train / 15% val / 15% test**, applying augmentation only to the training portion.

In [ ]:
# Cell 12: Custom Dataset wrapper so we can apply different transforms to train vs val
class CIFARSubset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, labels, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, _ = self.base_dataset[self.indices[i]]
        img = self.transform(img)
        label = self.labels[i]
        return img, label

# Split the train pool (train_idx/train_labels) into train (80%) and val (20%)
n_total = len(train_idx)
perm = np.random.RandomState(SEED).permutation(n_total)
n_val = int(0.2 * n_total)
val_pos, train_pos = perm[:n_val], perm[n_val:]

train_indices_final = [train_idx[i] for i in train_pos]
train_labels_final  = [train_labels[i] for i in train_pos]
val_indices_final   = [train_idx[i] for i in val_pos]
val_labels_final    = [train_labels[i] for i in val_pos]

train_ds = CIFARSubset(raw_train, train_indices_final, train_labels_final, train_transform)
val_ds   = CIFARSubset(raw_train, val_indices_final, val_labels_final, eval_transform)
test_ds  = CIFARSubset(raw_test, test_idx, test_labels, eval_transform)

print("Train:", len(train_ds), " Val:", len(val_ds), " Test:", len(test_ds))


In [ ]:
# Cell 13: DataLoaders (num_workers=0 -> avoids Colab CPU multiprocessing issues)
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("One batch shape:", xb.shape, "| labels shape:", yb.shape)


## 7. Convolution — The Core Idea (from scratch, no libraries)

A small matrix called a **filter/kernel** slides over the image, doing element-wise multiplication + summation at each position.

**Output size formula** (per spatial dimension):

```
Output = floor((Input - Kernel + 2*Padding) / Stride) + 1
```

Let's implement raw convolution in pure NumPy so the mechanics are 100% transparent before we let PyTorch do it for us.

In [ ]:
# Cell 14: Manual 2D convolution (no padding, stride 1) - purely educational
def conv2d_manual(image, kernel, stride=1, padding=0):
    if padding > 0:
        image = np.pad(image, pad_width=padding, mode="constant", constant_values=0)
    H, W = image.shape
    kH, kW = kernel.shape
    out_h = (H - kH) // stride + 1
    out_w = (W - kW) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(0, out_h):
        for j in range(0, out_w):
            region = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            output[i, j] = np.sum(region * kernel)
    return output

image = np.array([
    [1, 2, 3, 0, 1],
    [4, 5, 6, 1, 0],
    [7, 8, 9, 2, 1],
    [1, 0, 1, 3, 2],
    [0, 1, 2, 1, 4],
], dtype=float)

edge_kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
], dtype=float)

feature_map = conv2d_manual(image, edge_kernel, stride=1, padding=0)
print("Input shape:", image.shape)
print("Kernel shape:", edge_kernel.shape)
print("Output (feature map) shape:", feature_map.shape)
print(feature_map)

# Verify against the formula
H, kH, pad, stride = 5, 3, 0, 1
expected = (H - kH + 2*pad)//stride + 1
print("Formula-predicted output size:", expected, "-> matches:", expected == feature_map.shape[0])


In [ ]:
# Cell 15: Same idea with stride=2 and padding=1 (Same-ish padding)
feature_map_2 = conv2d_manual(image, edge_kernel, stride=2, padding=1)
print("With stride=2, padding=1 -> output shape:", feature_map_2.shape)
print(feature_map_2)


In [ ]:
# Cell 16: Max pooling & average pooling from scratch
def pool2d_manual(image, size=2, stride=2, mode="max"):
    H, W = image.shape
    out_h = (H - size)//stride + 1
    out_w = (W - size)//stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+size, j*stride:j*stride+size]
            output[i, j] = region.max() if mode == "max" else region.mean()
    return output

region_demo = np.array([[1, 5], [3, 2]], dtype=float)
print("2x2 region:\n", region_demo)
print("Max pooling ->", pool2d_manual(region_demo, size=2, stride=2, mode="max"))
print("Avg pooling ->", pool2d_manual(region_demo, size=2, stride=2, mode="avg"))


## 8. Building a CNN in PyTorch

Standard block pattern:

```
Input → Conv → BatchNorm → ReLU → Pool → Conv → BatchNorm → ReLU → Pool → GAP/Flatten → Dense → Output
```

We use **Global Average Pooling (GAP)** instead of Flatten — it uses far fewer parameters than flattening, which keeps the model small and fast on CPU.

In [ ]:
# Cell 17: Small, CPU-friendly CNN built from scratch
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, in_channels=3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /4
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /8
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling -> 1x1 regardless of input size
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)              # (B, 64, 1, 1)
        x = torch.flatten(x, 1)      # (B, 64)
        x = self.dropout(x)
        x = self.classifier(x)       # (B, num_classes)
        return x

model = SimpleCNN(num_classes=len(KEEP_CLASSES)).to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {n_params:,}  (small on purpose -> fast on CPU)")


In [ ]:
# Cell 18: Sanity-check forward pass and shapes
with torch.no_grad():
    out = model(xb.to(device))
print("Input batch shape:", xb.shape)
print("Output (logits) shape:", out.shape, "-> (batch_size, num_classes)")


## 9. Visualizing Filters and Feature Maps

Early layers typically learn simple patterns (edges, colors); this becomes clearer once the model is trained, but we can already look at the **raw first-layer filters** and the **feature maps they produce** on a real image.

In [ ]:
# Cell 19: Visualize first-conv-layer filters (before training, so they'll look random)
first_conv = model.block1[0]
filters = first_conv.weight.data.clone()  # shape (16, 3, 3, 3)
filters = (filters - filters.min()) / (filters.max() - filters.min() + 1e-8)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i].permute(1, 2, 0).numpy())
    ax.axis("off")
plt.suptitle("First-layer conv filters (untrained -> look random for now)")
plt.show()


In [ ]:
# Cell 20: Visualize feature maps produced by block1 on a real image
model.eval()
sample_x, sample_y = next(iter(val_loader))
with torch.no_grad():
    fmap = model.block1(sample_x.to(device))[0]  # first image in batch -> (16, H, W)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmap[i].cpu().numpy(), cmap="viridis")
    ax.axis("off")
plt.suptitle(f"16 feature maps from block1 (image class: {KEEP_CLASSES[sample_y[0]]})")
plt.show()


## 10. Loss Function & Optimizer

- **Multi-class classification** → Cross-Entropy Loss
- **Optimizer** → Adam (a strong default starting point)
- We also add **weight decay** (L2 regularization) and an LR scheduler to demonstrate good practice.

In [ ]:
# Cell 21: Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
print("Loss:", criterion)
print("Optimizer:", optimizer)


## 11. Training Loop (with early stopping)

Full training cycle per batch: **Forward pass → Loss → Backprop → Gradients → Optimizer step**.

We keep `EPOCHS` small and add **early stopping** so the notebook never runs unnecessarily long or "cracks down" (crashes/hangs) on CPU — it stops as soon as validation stops improving.

In [ ]:
# Cell 22: Train / validate one epoch (reusable functions)
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if is_train:
            optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total_samples += xb.size(0)

    torch.set_grad_enabled(True)
    return total_loss / total_samples, total_correct / total_samples


In [ ]:
# Cell 23: Full training with early stopping (CPU-safe: capped epochs + patience)
EPOCHS = 12
PATIENCE = 4

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
          f"| val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} (no improvement for {PATIENCE} epochs).")
            break

print(f"\nTraining finished in {time.time() - start_time:.1f}s on CPU.")
if best_state is not None:
    model.load_state_dict(best_state)
    print("Restored best model weights (lowest validation loss).")


In [ ]:
# Cell 24: Plot training curves -> diagnose overfitting/underfitting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()

gap = history["train_acc"][-1] - history["val_acc"][-1]
if gap > 0.15:
    print(f"Train/val accuracy gap = {gap:.2f} -> possible OVERFITTING. Consider more augmentation, dropout, or more data.")
elif history["train_acc"][-1] < 0.5:
    print("Both train and val accuracy are low -> possible UNDERFITTING. Consider a bigger model or more epochs.")
else:
    print(f"Train/val accuracy gap = {gap:.2f} -> looks reasonably healthy.")


## 12. Evaluation on the Test Set

We look at **accuracy, precision, recall, F1-score, and a confusion matrix** — never judge a classifier on accuracy alone, especially with imbalanced classes.

In [ ]:
# Cell 25: Final test evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Test accuracy: {test_acc:.3f}\n")
print(classification_report(all_labels, all_preds, target_names=KEEP_CLASSES))


In [ ]:
# Cell 26: Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(KEEP_CLASSES))); ax.set_xticklabels(KEEP_CLASSES, rotation=45)
ax.set_yticks(range(len(KEEP_CLASSES))); ax.set_yticklabels(KEEP_CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 27: Look at a few predictions (correct and incorrect)
model.eval()
xb, yb = next(iter(test_loader))
with torch.no_grad():
    preds = model(xb.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(xb):
        ax.axis("off"); continue
    img = xb[i] * std[:, None, None] + mean[:, None, None]
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    correct = preds[i].item() == yb[i].item()
    ax.set_title(f"true:{KEEP_CLASSES[yb[i]]}\npred:{KEEP_CLASSES[preds[i]]}",
                 color="green" if correct else "red", fontsize=9)
    ax.axis("off")
plt.suptitle("Green = correct, Red = wrong")
plt.tight_layout()
plt.show()


## 13. Interpretability — Feature Maps After Training

Now that the model is trained, first-layer filters and feature maps should look much more structured (edges, color blobs) than the random ones we saw before training.

In [ ]:
# Cell 28: Re-visualize trained filters + feature maps
first_conv = model.block1[0]
filters = first_conv.weight.data.clone()
filters = (filters - filters.min()) / (filters.max() - filters.min() + 1e-8)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i].permute(1, 2, 0).numpy())
    ax.axis("off")
plt.suptitle("First-layer filters AFTER training")
plt.show()

with torch.no_grad():
    fmap = model.block1(sample_x.to(device))[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmap[i].cpu().numpy(), cmap="viridis")
    ax.axis("off")
plt.suptitle(f"Feature maps AFTER training (class: {KEEP_CLASSES[sample_y[0]]})")
plt.show()


## 14. Advanced Building Block — Residual (Skip) Connections

**ResNet's** key idea: instead of a block learning a full mapping `H(x)`, it learns a **residual** `F(x) = H(x) - x`, and outputs `F(x) + x`. This "skip connection" makes very deep networks much easier to train by giving gradients a direct path backward.

```
x ───────────────┐
│                 │
↓                 (+)
Conv → BN → ReLU → Conv → BN ──┘
```

We implement one **residual block** and sanity-check its input/output shapes — you can drop this into `SimpleCNN` in place of a plain conv block for a deeper architecture.

In [ ]:
# Cell 29: A single residual block (shape-preserving) — architecture demo
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity   # <-- the skip connection
        return self.relu(out)

res_block = ResidualBlock(channels=16)
dummy = torch.randn(2, 16, 32, 32)
out = res_block(dummy)
print("Input shape: ", dummy.shape)
print("Output shape:", out.shape, "(same shape -> can be stacked repeatedly)")


## 15. Transfer Learning (the practical superpower of CNNs)

Instead of training from scratch, we reuse a model **pretrained on ImageNet** and adapt it to our 4 classes.

We use **MobileNetV2** — it's specifically designed to be lightweight (depthwise separable convolutions), so it's one of the few pretrained CNNs that's actually reasonable to fine-tune on a Colab **CPU**.

Two stages:
1. **Feature extraction** — freeze the backbone, train only a new classifier head
2. **Fine-tuning** (optional) — unfreeze the last few layers with a tiny learning rate

In [ ]:
# Cell 30: Load pretrained MobileNetV2, freeze backbone, replace head
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

for param in mobilenet.features.parameters():
    param.requires_grad = False   # freeze backbone

in_features = mobilenet.classifier[1].in_features
mobilenet.classifier[1] = nn.Linear(in_features, len(KEEP_CLASSES))
mobilenet = mobilenet.to(device)

trainable = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
total = sum(p.numel() for p in mobilenet.parameters())
print(f"Trainable params: {trainable:,} / {total:,} total (backbone frozen)")


In [ ]:
# Cell 31: MobileNetV2 needs its own preprocessing (ImageNet mean/std, min ~96px input)
MOBILENET_SIZE = 96  # small side for CPU speed; MobileNetV2 handles this fine via global pooling

mobilenet_train_tf = transforms.Compose([
    transforms.Resize((MOBILENET_SIZE, MOBILENET_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
mobilenet_eval_tf = transforms.Compose([
    transforms.Resize((MOBILENET_SIZE, MOBILENET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

mnet_train_ds = CIFARSubset(raw_train, train_indices_final, train_labels_final, mobilenet_train_tf)
mnet_val_ds   = CIFARSubset(raw_train, val_indices_final, val_labels_final, mobilenet_eval_tf)
mnet_test_ds  = CIFARSubset(raw_test, test_idx, test_labels, mobilenet_eval_tf)

mnet_train_loader = DataLoader(mnet_train_ds, batch_size=32, shuffle=True,  num_workers=0)
mnet_val_loader   = DataLoader(mnet_val_ds,   batch_size=32, shuffle=False, num_workers=0)
mnet_test_loader  = DataLoader(mnet_test_ds,  batch_size=32, shuffle=False, num_workers=0)
print("MobileNet-ready loaders created.")


In [ ]:
# Cell 32: Train just the new classifier head (few epochs -> fast, since backbone is frozen)
mnet_criterion = nn.CrossEntropyLoss()
mnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet.parameters()), lr=1e-3
)

MNET_EPOCHS = 4
mnet_history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

start_time = time.time()
for epoch in range(1, MNET_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(mobilenet, mnet_train_loader, mnet_criterion, mnet_optimizer)
    va_loss, va_acc = run_epoch(mobilenet, mnet_val_loader, mnet_criterion, optimizer=None)
    mnet_history["train_loss"].append(tr_loss); mnet_history["val_loss"].append(va_loss)
    mnet_history["train_acc"].append(tr_acc); mnet_history["val_acc"].append(va_acc)
    print(f"Epoch {epoch}/{MNET_EPOCHS} | train_acc={tr_acc:.3f} val_acc={va_acc:.3f}")

print(f"\nTransfer learning finished in {time.time() - start_time:.1f}s on CPU.")


In [ ]:
# Cell 33: Evaluate the transfer-learning model on the test set
mobilenet.eval()
mnet_preds, mnet_labels = [], []
with torch.no_grad():
    for xb, yb in mnet_test_loader:
        outputs = mobilenet(xb.to(device))
        mnet_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        mnet_labels.extend(yb.numpy())

mnet_test_acc = (np.array(mnet_preds) == np.array(mnet_labels)).mean()
print(f"MobileNetV2 (transfer learning) test accuracy: {mnet_test_acc:.3f}")
print(f"SimpleCNN (trained from scratch) test accuracy: {test_acc:.3f}")
print("\nTransfer learning usually wins with far less training, because the backbone")
print("already knows general visual features (edges, textures, shapes) from ImageNet.")


### Optional: Fine-tuning
Unfreeze the last block of MobileNetV2 and train a bit more with a **very small learning rate** — this squeezes out extra accuracy without destroying the pretrained features. Skip this cell if you're short on time; it's optional.

In [ ]:
# Cell 34: (Optional) Fine-tune last few layers with a tiny LR
FINE_TUNE = False  # flip to True if you have a few extra minutes

if FINE_TUNE:
    for param in mobilenet.features[-3:].parameters():
        param.requires_grad = True

    fine_tune_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, mobilenet.parameters()), lr=1e-5
    )
    for epoch in range(1, 3):
        tr_loss, tr_acc = run_epoch(mobilenet, mnet_train_loader, mnet_criterion, fine_tune_optimizer)
        va_loss, va_acc = run_epoch(mobilenet, mnet_val_loader, mnet_criterion, optimizer=None)
        print(f"Fine-tune epoch {epoch} | train_acc={tr_acc:.3f} val_acc={va_acc:.3f}")
else:
    print("FINE_TUNE=False -> skipped (set to True to try it).")


## 16. Saving, Loading, and Running Inference

A trained model is only useful if you can save it and reload it later for predictions.

In [ ]:
# Cell 35: Save both models
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/simple_cnn.pt")
torch.save(mobilenet.state_dict(), "checkpoints/mobilenet_transfer.pt")
print("Saved: checkpoints/simple_cnn.pt")
print("Saved: checkpoints/mobilenet_transfer.pt")


In [ ]:
# Cell 36: Reload and do a quick sanity check
reloaded = SimpleCNN(num_classes=len(KEEP_CLASSES))
reloaded.load_state_dict(torch.load("checkpoints/simple_cnn.pt", map_location="cpu"))
reloaded.eval()

with torch.no_grad():
    xb, yb = next(iter(test_loader))
    out = reloaded(xb)
    preds = out.argmax(dim=1)
acc_check = (preds == yb).float().mean().item()
print(f"Reloaded model accuracy on one test batch: {acc_check:.3f}")


In [ ]:
# Cell 37: Single-image inference function (drop in any PIL image)
def predict_image(pil_image, model, transform, class_names):
    model.eval()
    x = transform(pil_image).unsqueeze(0)  # add batch dim
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
    return class_names[pred_idx], probs[pred_idx].item()

# Try it on a test image
sample_pos = 0
test_img, _ = raw_test[test_idx[sample_pos]]
true_class = KEEP_CLASSES[test_labels[sample_pos]]
pred_class, confidence = predict_image(test_img, reloaded, eval_transform, KEEP_CLASSES)

plt.imshow(test_img)
plt.title(f"Predicted: {pred_class} ({confidence:.1%}) | True: {true_class}")
plt.axis("off")
plt.show()


## 17. CNN Hyperparameter Cheat Sheet

| Hyperparameter | What it controls | Typical starting point |
|---|---|---|
| Number of filters | How many patterns a layer can detect | 16 → 32 → 64 → 128 (increasing with depth) |
| Kernel size | Receptive field per layer | 3×3 (most common) |
| Stride | How far the filter moves | 1 (or 2 for downsampling) |
| Padding | Whether spatial size shrinks | "same" to preserve size |
| Pooling size/stride | Downsampling amount | 2×2, stride 2 |
| Learning rate | Step size of weight updates | 1e-3 (Adam) |
| Batch size | Images per gradient step | 32–64 on CPU |
| Dropout | Regularization strength | 0.2–0.5 |
| Weight decay | L2 regularization strength | 1e-4–1e-5 |
| Epochs | Full passes over training data | Use early stopping instead of guessing |

## 18. CNN vs. Other Architectures

| | Fully Connected NN | CNN | RNN | Vision Transformer |
|---|---|---|---|---|
| Best for | Tabular data | Images (spatial patterns) | Sequences (text, time series) | Images (global attention) |
| Key mechanism | Dense connections | Local connectivity + parameter sharing | Sequential hidden state | Self-attention |
| Params for images | Huge | Efficient (shared filters) | N/A | Large, but flexible |
| Inductive bias | None | Locality + translation | Temporal order | Minimal (needs more data) |

In [ ]:
# Cell 38: Quick param-count comparison: FC vs CNN for a 64x64x3 image, 4 classes
fc_equivalent_params = (64*64*3) * 128 + 128 * 4   # one hidden layer of 128 units, for comparison
cnn_params = sum(p.numel() for p in model.parameters())

print(f"A single dense layer with 128 hidden units on a 64x64x3 image alone needs "
      f"~{fc_equivalent_params:,} parameters.")
print(f"Our full CNN (3 conv blocks + classifier) uses only {cnn_params:,} parameters.")
print("This is the power of local connectivity + parameter sharing.")


## 19. Complete CNN Mastery Checklist

Everything below was covered hands-on in this notebook:

**Fundamentals** ✅ What is a CNN · Image representation · Channels (RGB/grayscale) · Feature extraction

**Data** ✅ Ingestion · EDA · Corrupted-image checks · Resizing · Normalization · Augmentation · Train/val/test split

**Convolution** ✅ Kernel/filter · Convolution operation (from scratch) · Feature maps · Stride · Padding · Output-size formula · Multiple filters · Parameter sharing · Local connectivity

**Pooling** ✅ Max pooling · Average pooling · Global Average Pooling (from scratch + in PyTorch)

**Network** ✅ ReLU · Softmax · Dense/classifier layers · Flatten vs GAP · Cross-entropy loss · Backpropagation (via autograd) · Adam optimizer

**Training** ✅ Batches · Epochs · Learning rate · LR scheduling (`ReduceLROnPlateau`) · Batch normalization · Dropout · Weight decay · Early stopping

**Diagnostics** ✅ Overfitting vs underfitting · Loss/accuracy curves · Confusion matrix · Precision/recall/F1

**Advanced concepts** ✅ Receptive field (conceptually, via stacked convs) · Residual/skip connections (implemented) · Parameter-count comparison vs fully-connected

**Architectures (reference)** ✅ LeNet, AlexNet, VGG, Inception/GoogLeNet, ResNet, DenseNet, MobileNet, EfficientNet — MobileNetV2 used hands-on for transfer learning

**Transfer Learning** ✅ Feature extraction (frozen backbone) · Fine-tuning (optional unfreeze) · Comparing scratch-trained vs pretrained accuracy

**Interpretability** ✅ Visualizing filters and feature maps before/after training

### What's deliberately out of scope here (natural next projects)
- **Object detection** (YOLO, Faster R-CNN, SSD) — outputs boxes, not just a class
- **Segmentation** (U-Net, Mask R-CNN, DeepLab) — pixel-level labeling
- **Vision Transformers** — attention-based alternative to convolution
- These build directly on everything you just did — the CNN backbone stays central.

### The big picture
```
Perceptron → Neural Network → CNN → Computer Vision
                              │
                              ├── RNN/LSTM → Sequential Data / NLP
                              └── Transformer → NLP / Vision / Multimodal → LLMs
```
Once you're comfortable with everything above, you have the real foundation for deep learning and generative AI — not just CNN theory, but a working, debuggable, from-scratch-to-transfer-learning pipeline.


In [ ]:
# Cell 39: Final summary printout
print("="*60)
print("PROJECT SUMMARY")
print("="*60)
print(f"Classes used        : {KEEP_CLASSES}")
print(f"Train / Val / Test   : {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")
print(f"SimpleCNN params     : {sum(p.numel() for p in model.parameters()):,}")
print(f"SimpleCNN test acc   : {test_acc:.3f}")
print(f"MobileNetV2 test acc : {mnet_test_acc:.3f}")
print(f"Device used          : {device} (fully CPU-friendly)")
print("="*60)
